In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, ShuffleSplit
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [3]:


# ===============================
# 2️⃣ LOAD DATA
# ===============================
# Use raw string for Windows path to avoid escape errors
# df = pd.read_csv(r"D:\PRICE_DS\bangalore price\antenna_data_ready.csv")
df = pd.read_csv(r"D:\PRICE_DS\bangalore price\antenna_data_ready.csv")


# Features and target
X = df[['patch_length_mm', 'patch_width_mm', 'substrate_h_mm', 'eps_r', 'feed_offset_mm']]
y = df['resonant_freq_GHz']  # Predict resonant frequency
# y = df['return_loss_dB']    # Uncomment if predicting return loss instead

# ===============================
# 3️⃣ SCALE FEATURES
# ===============================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# ===============================
# 4️⃣ DEFINE MODELS AND PARAMETERS
# ===============================
algos = {
    'Linear Regression': {
        'model': LinearRegression(),
        'params': {'fit_intercept': [True, False]}
    },
    'Lasso Regression': {
        'model': Lasso(max_iter=5000),
        'params': {'alpha': [0.1, 0.5, 1, 2], 'selection': ['random', 'cyclic']}
    },
    'Decision Tree': {
        'model': DecisionTreeRegressor(random_state=42),
        'params': {
            'criterion': ['squared_error', 'friedman_mse'],
            'splitter': ['best','random'],
            'max_depth': [None, 4, 6, 8]
        }
    },
    'Random Forest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [4, 6, 8, None],
            'min_samples_split': [2, 5, 10]
        }
    }
}

# ===============================
# 5️⃣ RUN GRIDSEARCHCV AND EVALUATE
# ===============================
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
results = []

for name, config in algos.items():
    # GridSearchCV
    model = GridSearchCV(config['model'], config['params'], cv=cv, scoring='r2', return_train_score=False)
    model.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = model.predict(X_test)
    
    # Store results
    results.append({
        'Model': name,
        'Best Hyperparameters': model.best_params_,
        'CV R² Score': round(model.best_score_, 4),
        'Test R²': round(r2_score(y_test, y_pred), 4),
        'Test MSE': round(mean_squared_error(y_test, y_pred), 4)
    })

# ===============================
# 6️⃣ DISPLAY RESULTS
# ===============================
results_df = pd.DataFrame(results)
print("=== Model Comparison ===")
display(results_df)

# Find overall best model based on Test R²
best_model_idx = results_df['Test R²'].idxmax()
best_model_info = results_df.loc[best_model_idx]

print("\n=== Best Model Summary ===")
print("Model:", best_model_info['Model'])
print("Best Hyperparameters:", best_model_info['Best Hyperparameters'])
print("CV R² Score:", best_model_info['CV R² Score'])
print("Test R²:", best_model_info['Test R²'])
print("Test MSE:", best_model_info['Test MSE'])


=== Model Comparison ===


,Model,Best Hyperparameters,CV R² Score,Test R²,Test MSE
0,Linear Regression,{'fit_intercept': True},0.9779,0.9859,0.0005
1,Lasso Regression,"{'alpha': 0.1, 'selection': 'random'}",0.7030,0.6894,0.0106
2,Decision Tree,"{'criterion': 'friedman_mse', 'max_depth': 8, ...",0.9707,0.9835,0.0006
3,Random Forest,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.9814,0.9879,0.0004



=== Best Model Summary ===
Model: Random Forest
Best Hyperparameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
CV R² Score: 0.9814
Test R²: 0.9879
Test MSE: 0.0004


In [4]:

FEATURES = [
    'patch_length_mm',
    'patch_width_mm',
    'substrate_h_mm',
    'eps_r',
    'feed_offset_mm'
]

TARGET = 'resonant_freq_GHz'
TARGET_RL = 'return_loss_dB'

X = df[FEATURES]
# y = df[TARGET]
y = df[TARGET_RL]

# -------------------------------
# Train / Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------
# Random Forest Model (BEST PARAMS)
# -------------------------------
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=None,
    min_samples_split=2,
    random_state=42
)

# Train
rf_model.fit(X_train, y_train)

# -------------------------------
# Evaluation
# -------------------------------
y_pred = rf_model.predict(X_test)

print("Test R²:", round(r2_score(y_test, y_pred), 4))
print("Test MSE:", round(mean_squared_error(y_test, y_pred), 6))


Test R²: 0.6989
Test MSE: 4.890187


In [7]:
print(X.columns)
print("Number of features:", X.shape[1])


Index(['patch_length_mm', 'patch_width_mm', 'substrate_h_mm', 'eps_r',
       'feed_offset_mm'],
      dtype='object')
Number of features: 5


In [8]:
import pandas as pd

new_design = pd.DataFrame([{
    'patch_length_mm': 28.5,
    'patch_width_mm': 34.2,
    'substrate_h_mm': 1.6,
    'eps_r': 4.4,
    'feed_offset_mm': 6.0
}])

predicted_freq = rf_model.predict(new_design)

# print("Predicted Resonant Frequency (GHz):", round(predicted_freq[0], 4))
print("Predicted Return Loss (dB):", round(predicted_freq[0], 4))


Predicted Return Loss (dB): -26.09
